# Phase 3 — LegalIR strong rerankers on RTX Pro 6000 (offline)
Attach competition data, the Phase 2 Harrier bundle, and the Phase 3 reranker delta bundle. Set Internet Off.

In [ ]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from pathlib import Path
EXPERIMENT_ID = 'phase3-rerankers-harrier-retrieval'
DATASET_DIR = Path('/kaggle/input/REPLACE_WITH_COMPETITION_DATASET_SLUG')
PHASE2_BUNDLE = Path('/kaggle/input/REPLACE_WITH_PHASE2_HARRIER_BUNDLE_SLUG/legalir-phase2-harrier-bundle')
DELTA_BUNDLE = Path('/kaggle/input/REPLACE_WITH_PHASE3_RERANKER_DELTA_SLUG/legalir-phase3-reranker-delta')
WORK_DIR = Path('/kaggle/working/legalir-phase3-run')
RUNTIME_DIR = Path('/kaggle/working/legalir-phase3-runtime')

In [ ]:
import json, shutil, subprocess, sys, time
def run(*command, cwd=None, env=None):
    print('+', ' '.join(map(str, command)))
    started = time.perf_counter()
    subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)
    print(f'Completed in {(time.perf_counter() - started) / 60:.1f} minutes')
delta_manifest = json.loads((DELTA_BUNDLE / 'manifests' / 'bundle_manifest.json').read_text(encoding='utf-8'))
if delta_manifest.get('experiment_id') != EXPERIMENT_ID: raise RuntimeError(f"Wrong Phase 3 bundle: {delta_manifest.get('experiment_id')}")
phase2_manifest_path = PHASE2_BUNDLE / 'manifests' / 'bundle_manifest.json'
if not phase2_manifest_path.is_file(): raise FileNotFoundError(f'Missing Phase 2 manifest: {phase2_manifest_path}')
phase2_manifest = json.loads(phase2_manifest_path.read_text(encoding='utf-8'))
phase2_names = {row['name'] for row in phase2_manifest['models']}
if not {'vietlegal_harrier', 'vietnamese_embedding', 'nemotron'} <= phase2_names: raise RuntimeError(f'Phase 2 bundle lacks retrievers: {phase2_names}')
for record in delta_manifest['files']:
    path = DELTA_BUNDLE / record['path']
    if not path.is_file() or path.stat().st_size != record['bytes']: raise RuntimeError(f'Missing/truncated delta file: {path}')
if WORK_DIR.exists(): shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)
if RUNTIME_DIR.exists(): shutil.rmtree(RUNTIME_DIR)
RUNTIME_DIR.mkdir(parents=True)
run(sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--ignore-installed', '--target', RUNTIME_DIR, '--find-links', DELTA_BUNDLE / 'wheels', '-r', DELTA_BUNDLE / 'requirements-offline.txt')
project_wheels = sorted((DELTA_BUNDLE / 'wheels').glob('uit_legalir-*.whl'))
if len(project_wheels) != 1: raise RuntimeError(f'Expected one project wheel, found {project_wheels}')
run(sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--ignore-installed', '--target', RUNTIME_DIR, project_wheels[0])
runtime_env = os.environ.copy()
runtime_env['PYTHONPATH'] = str(RUNTIME_DIR)
runtime_env['PYTHONNOUSERSITE'] = '1'
print('Phase 3 delta commit:', delta_manifest['project_commit'])

In [ ]:
gpu_probe = "import torch; assert torch.cuda.is_available(), 'No CUDA GPU available'; print('GPU:', torch.cuda.get_device_name(0)); print('CUDA:', torch.version.cuda)"
run(sys.executable, '-c', gpu_probe, env=runtime_env)
contexts_source = DATASET_DIR / 'selected-contexts' / 'selected-contexts'
if not contexts_source.is_dir(): raise FileNotFoundError(f'Missing corpus: {contexts_source}')
for filename in ('train.json', 'public-official.json'):
    if not (DATASET_DIR / filename).is_file(): raise FileNotFoundError(f'Missing input: {filename}')
(WORK_DIR / 'selected-contexts').symlink_to(contexts_source, target_is_directory=True)
for filename in ('train.json', 'public-official.json'): (WORK_DIR / filename).symlink_to(DATASET_DIR / filename)
print('Contexts:', sum(1 for _ in contexts_source.glob('context_*.json')))

In [ ]:
import yaml
config = yaml.safe_load((DELTA_BUNDLE / 'configs' / 'kaggle_rtx_pro_6000.yaml').read_text(encoding='utf-8'))
for name in ('vietlegal_harrier', 'vietnamese_embedding', 'nemotron'): config['models'][name]['local_path'] = str(PHASE2_BUNDLE / 'models' / name)
for name in ('legal_reranker', 'qwen3_reranker', 'prism_reranker'): config['models'][name]['local_path'] = str(DELTA_BUNDLE / 'models' / name)
for spec in config['models'].values(): spec['local_files_only'] = True
config_path = WORK_DIR / 'kaggle_rtx_pro_6000_phase3.yaml'
config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding='utf-8')
print(config_path.read_text(encoding='utf-8'))

In [ ]:
preflight = WORK_DIR / 'phase3_preflight.py'
preflight.write_text('''
import sys
from pathlib import Path
import torch
import yaml
from legalir.embeddings import load_encoder
from legalir.rerank import PairwiseReranker, CausalYesNoReranker
config = yaml.safe_load(Path(sys.argv[1]).read_text(encoding='utf-8'))
for name, spec in config['models'].items():
    if spec['role'] == 'dense':
        model = load_encoder(spec, config['runtime'])
        assert len(model.encode([spec['prompt_query'] + 'điều kiện cấp giấy phép'], convert_to_numpy=True)) == 1
        del model
        torch.cuda.empty_cache()
for name, spec in config['models'].items():
    if spec['role'] == 'pairwise_reranker': engine = PairwiseReranker(config, name)
    elif spec['role'] == 'causal_reranker': engine = CausalYesNoReranker(config, name)
    else: continue
    assert len(engine.rank('câu hỏi pháp luật', ['văn bản pháp luật liên quan', 'văn bản không liên quan'])) == 2
    engine.close()
print('Phase 3 all local model preflight tests passed.')
'''.lstrip(), encoding='utf-8')
run(sys.executable, preflight, config_path, cwd=WORK_DIR, env=runtime_env)

In [ ]:
artifacts = WORK_DIR / config['paths']['artifacts_dir']
artifacts.mkdir(parents=True, exist_ok=True)
first_stage = {'weights': {'bm25': 0.0, 'accent_char': 0.5, 'vietlegal_harrier': 2.0, 'vietnamese_embedding': 1.0, 'nemotron': 2.0, 'query_memory': 1.0, 'query_exact': 4.0}, 'rrf_k': 20}
(artifacts / 'first_stage_weights.json').write_text(json.dumps(first_stage, ensure_ascii=False, indent=2), encoding='utf-8')
base = [sys.executable, '-m', 'legalir']
def legalir(*args): run(*base, *args, cwd=WORK_DIR, env=runtime_env)
legalir('prepare', '--config', config_path, '--resume')
legalir('audit', '--config', config_path)
legalir('index', '--config', config_path, '--lexical-only', '--resume')
for model in ('vietlegal_harrier', 'vietnamese_embedding', 'nemotron'): legalir('index', '--config', config_path, '--model', model, '--resume')
legalir('tune', '--config', config_path, '--final', '--fold', '0', '--resume')
shutil.copy2(artifacts / 'final_weights_fold0.json', artifacts / 'final_weights.json')
legalir('retrieve', '--config', config_path, '--split', 'public', '--resume')
for engine in ('legal_reranker', 'qwen3_reranker', 'prism_reranker'): legalir('rerank', '--config', config_path, '--split', 'public', '--engine', engine, '--resume')
legalir('rerank', '--config', config_path, '--split', 'public', '--resume')
legalir('predict', '--config', config_path, '--output', WORK_DIR / 'submission_phase3_tuned.json', '--resume')
run('zip', '-j', WORK_DIR / 'submission_phase3_tuned.zip', WORK_DIR / 'submission_phase3_tuned.json')
print('Submission:', WORK_DIR / 'submission_phase3_tuned.zip')

In [ ]:
manifest = json.loads((artifacts / 'model_manifest.json').read_text(encoding='utf-8'))
final_stage = json.loads((artifacts / 'final_weights.json').read_text(encoding='utf-8'))
report = {'experiment_id': EXPERIMENT_ID, 'project_commit': delta_manifest['project_commit'], 'models': manifest['models'], 'total_parameters': manifest['total_parameters'], 'first_stage': first_stage, 'final_stage': final_stage, 'submission': str(WORK_DIR / 'submission_phase3_tuned.zip')}
(WORK_DIR / 'phase3_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(report, ensure_ascii=False, indent=2))